## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from joblib import dump
from paths import PROCESSED_DIR, MODEL_DIR, RESULT_DIR

## Load the data file

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
test = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
y_train = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')['reordered']
X_test = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
y_test = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet')['reordered']

In [ ]:
X_test

## Train the Random Forest model

In [ ]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

In [ ]:
model.fit(X_train, y_train)

## Make predictions on the test set and calculate evaluation metrics

In [ ]:
test_pred_prob = model.predict_proba(X_test)[:, 1]
test_pred_05 = (test_pred_prob >= 0.5).astype(int)

print('AUC:       ', round(roc_auc_score(y_test, test_pred_prob), 4))
print('PR-AUC:    ', round(average_precision_score(y_test, test_pred_prob), 4))
print('F1@0.50:   ', round(f1_score(y_test, test_pred_05, zero_division=0), 4))
print('Precision: ', round(precision_score(y_test, test_pred_05, zero_division=0), 4))
print('Recall:    ', round(recall_score(y_test, test_pred_05, zero_division=0), 4))

## Search for the optimal classification threshold

In [ ]:
threshold_results = []

for th in np.arange(0.10, 0.91, 0.05):
    pred = (test_pred_prob >= th).astype(int)
    threshold_results.append({
        'threshold': round(float(th), 2),
        'f1': f1_score(y_test, pred, zero_division=0),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results).sort_values('f1', ascending=False)
threshold_df.head(10)

## Select the threshold with the best F1-score from the results above

In [ ]:
best_threshold = float(threshold_df.iloc[0]['threshold'])
best_f1 = float(threshold_df.iloc[0]['f1'])

best_pred = (test_pred_prob >= best_threshold).astype(int)

print('Best threshold:', best_threshold)
print('Best F1:       ', round(best_f1, 4))
print('Precision:     ', round(precision_score(y_test, best_pred, zero_division=0), 4))
print('Recall:        ', round(recall_score(y_test, best_pred, zero_division=0), 4))

## Calculate feature importance

In [ ]:
X_train.columns

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df

## Plot the top 15 most important features as a horizontal bar chart

In [ ]:
plt.figure(figsize=(8, 4))
top_imp = importance_df.head(15).sort_values('importance')
plt.barh(top_imp['feature'], top_imp['importance'])
plt.title('Top Feature Importances')
plt.xlabel('Importance')
plt.show()

## Combine ground truth and predictions into a table

In [ ]:
test_result = test[['user_id', 'product_id', 'reordered']].copy()
test_result['pred_prob'] = test_pred_prob
test_result['pred_label_0.50'] = test_pred_05
test_result['pred_label_best'] = best_pred

test_result.head()

## Save the test predictions and the trained Random Forest model

In [ ]:
MODEL_PATH = MODEL_DIR / 'random_forest_model.joblib'
RESULT_PATH = RESULT_DIR / 'test_predictions_random_forest.csv'
test_result.to_csv(RESULT_PATH, index=False)
dump(model, MODEL_PATH)
print('Saved:', RESULT_PATH.resolve())
print('Saved:', MODEL_PATH.resolve())

## Predict for a single user-product pair

In [ ]:
def predict_reorder(user_id, product_id, feature_table, trained_model, feature_columns, threshold):
    row = feature_table[(feature_table['user_id'] == user_id) & (feature_table['product_id'] == product_id)]
    if row.empty:
        return f'No existing features found for user_id={user_id}, product_id={product_id}'
    x = row[feature_columns]
    prob = trained_model.predict_proba(x)[:, 1][0]
    label = int(prob >= threshold)
    return {
        'user_id': int(user_id),
        'product_id': int(product_id),
        'pred_prob': float(prob),
        'pred_label': label,
        'threshold': float(threshold)
    }